In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
# Install evaluation library
!pip -q install evaluate jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 81.3 MB/s eta 0:00:00:00:01


In [3]:
# ============================================================
# PART 1 - IMPORTS
# ============================================================

import os
import re
import json
import random
import warnings
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
import evaluate

from transformers import (
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
    TrainingArguments,
    Trainer
)

from dataclasses import dataclass
from typing import Dict, List, Union

warnings.filterwarnings("ignore")

# ============================================================
# SEED
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 70)
print("DEVICE :", device)

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

print("=" * 70)

# ============================================================
# MODEL
# ============================================================

MODEL_NAME = "facebook/wav2vec2-base-960h"

print("Model :", MODEL_NAME)

print("=" * 70)

DEVICE : cuda
GPU : Tesla T4
Model : facebook/wav2vec2-base-960h


In [4]:
# ============================================================
# PART 2 - LOAD FLEURS ENGLISH DATASET
# ============================================================

print("=" * 70)
print("LOADING FLEURS ENGLISH DATASET")
print("=" * 70)

# Load English split directly
dataset = load_dataset(
    "google/fleurs",
    "en_us",
    trust_remote_code=True
)

print(dataset)

print("\n" + "=" * 70)
print("DATASET INFORMATION")
print("=" * 70)

print("Train Samples      :", len(dataset["train"]))
print("Validation Samples :", len(dataset["validation"]))
print("Test Samples       :", len(dataset["test"]))

print("\nFeatures:")
print(dataset["train"].column_names)

print("\nExample Transcription:")
print(dataset["train"][0]["transcription"])

print("=" * 70)

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'google/fleurs' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


LOADING FLEURS ENGLISH DATASET


README.md: 0.00B [00:00, ?B/s]

parquet-data/en_us/train-00000-of-00001.(…):   0%|          | 0.00/1.72G [00:00<?, ?B/s]

parquet-data/en_us/validation-00000-of-0(…):   0%|          | 0.00/237M [00:00<?, ?B/s]

parquet-data/en_us/test-00000-of-00001.p(…):   0%|          | 0.00/402M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2602 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/394 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/647 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id'],
        num_rows: 2602
    })
    validation: Dataset({
        features: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id'],
        num_rows: 394
    })
    test: Dataset({
        features: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id'],
        num_rows: 647
    })
})

DATASET INFORMATION
Train Samples      : 2602
Validation Samples : 394
Test Samples       : 647

Features:
['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id']

Example Transcription:
a tornado is a spinning column of very low-pressure air which sucks the surrounding air inward and upward


In [5]:
# ============================================================
# PART 3 - VERIFY ENGLISH DATASET
# ============================================================

print("=" * 70)
print("VERIFY ENGLISH DATASET")
print("=" * 70)

train_dataset = dataset["train"]
validation_dataset = dataset["validation"]
test_dataset = dataset["test"]

print("Train Samples      :", len(train_dataset))
print("Validation Samples :", len(validation_dataset))
print("Test Samples       :", len(test_dataset))

print("\nDataset Features:")
print(train_dataset.column_names)

print("\nSample Information")
print("-" * 70)

sample = train_dataset[0]

print("Audio Path:")
print(sample["path"])

print("\nTranscription:")
print(sample["transcription"])

print("\nSampling Rate:")
print(sample["audio"]["sampling_rate"])

print("\nAudio Length (samples):")
print(len(sample["audio"]["array"]))

print("=" * 70)

VERIFY ENGLISH DATASET
Train Samples      : 2602
Validation Samples : 394
Test Samples       : 647

Dataset Features:
['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id']

Sample Information
----------------------------------------------------------------------
Audio Path:
/root/.cache/huggingface/datasets/downloads/extracted/be467d88ba270014363a9d0aaae3893b4701a2710e0a55c6091a6d4fa56a9d84/10004088536354799741.wav

Transcription:
a tornado is a spinning column of very low-pressure air which sucks the surrounding air inward and upward

Sampling Rate:
16000

Audio Length (samples):
108800


In [6]:
# ============================================================
# PART 4 - LOAD PROCESSOR & MODEL
# ============================================================

import torch
from transformers import (
    Wav2Vec2Processor,
    Wav2Vec2ForCTC
)

print("=" * 70)
print("LOADING PROCESSOR & MODEL")
print("=" * 70)

MODEL_NAME = "facebook/wav2vec2-base-960h"

# ------------------------------------------------------------
# Processor
# ------------------------------------------------------------

processor = Wav2Vec2Processor.from_pretrained(
    MODEL_NAME
)

print("✓ Processor Loaded")

# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id
)

# Freeze CNN Feature Extractor
model.freeze_feature_encoder()

# Device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)

print("✓ Model Loaded")
print("Device :", device)

print("=" * 70)

# ============================================================
# VERIFY
# ============================================================

print("Model Vocabulary :", model.config.vocab_size)
print("Tokenizer Vocabulary :", len(processor.tokenizer))

sample_text = train_dataset[0]["transcription"]

print("\nSample Text:")
print(sample_text)

# Convert to uppercase because facebook/wav2vec2-base-960h
# tokenizer is trained with uppercase characters
token_ids = processor(
    text=sample_text.upper(),
    add_special_tokens=False
).input_ids

print("\nFirst 30 Token IDs:")
print(token_ids[:30])

decoded = processor.decode(token_ids)

print("\nDecoded Text:")
print(decoded)

print("=" * 70)

LOADING PROCESSOR & MODEL


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

...

✓ Processor Loaded


model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-960h
Key                        | Status  | 
---------------------------+---------+-
wav2vec2.masked_spec_embed | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✓ Model Loaded
Device : cuda
Model Vocabulary : 32
Tokenizer Vocabulary : 32

Sample Text:
a tornado is a spinning column of very low-pressure air which sucks the surrounding air inward and upward

First 30 Token IDs:
[7, 4, 6, 8, 13, 9, 7, 14, 8, 4, 10, 12, 4, 7, 4, 12, 23, 10, 9, 9, 10, 9, 21, 4, 19, 8, 15, 16, 17, 9]

Decoded Text:
A TORNADO IS A SPINING COLUMN OF VERY LOW<unk>PRESURE AIR WHICH SUCKS THE SUROUNDING AIR INWARD AND UPWARD


In [7]:
# ============================================================
# PART 3 - BUILD VOCABULARY
# ============================================================

import json
import re

print("=" * 70)
print("BUILDING VOCABULARY")
print("=" * 70)

# Convert Column -> List
train_text = list(dataset["train"]["transcription"])
valid_text = list(dataset["validation"]["transcription"])
test_text  = list(dataset["test"]["transcription"])

# Merge all transcripts
all_text = " ".join(train_text + valid_text + test_text)

# Lowercase
all_text = all_text.lower()

# Remove everything except a-z, apostrophe and space
all_text = re.sub(r"[^a-z' ]", " ", all_text)

# Create vocabulary
vocab_list = sorted(set(all_text))

vocab_dict = {v: k for k, v in enumerate(vocab_list)}

# Replace space with |
vocab_dict["|"] = vocab_dict[" "]
del vocab_dict[" "]

# Add special tokens
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

# Save vocabulary
with open("vocab.json", "w") as vocab_file:
    json.dump(vocab_dict, vocab_file)

print("Vocabulary Size :", len(vocab_dict))
print(vocab_dict)
print("=" * 70)

BUILDING VOCABULARY
Vocabulary Size : 30
{"'": 1, 'a': 2, 'b': 3, 'c': 4, 'd': 5, 'e': 6, 'f': 7, 'g': 8, 'h': 9, 'i': 10, 'j': 11, 'k': 12, 'l': 13, 'm': 14, 'n': 15, 'o': 16, 'p': 17, 'q': 18, 'r': 19, 's': 20, 't': 21, 'u': 22, 'v': 23, 'w': 24, 'x': 25, 'y': 26, 'z': 27, '|': 0, '[UNK]': 28, '[PAD]': 29}


In [8]:
# ============================================================
# PART 4 - LOAD TOKENIZER, PROCESSOR & MODEL
# ============================================================

import torch

from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
    Wav2Vec2ForCTC
)

print("=" * 70)
print("CREATING TOKENIZER, PROCESSOR & MODEL")
print("=" * 70)

# ------------------------------------------------------------
# MODEL NAME
# ------------------------------------------------------------

MODEL_NAME = "facebook/wav2vec2-base"

# ------------------------------------------------------------
# TOKENIZER
# ------------------------------------------------------------

tokenizer = Wav2Vec2CTCTokenizer(
    "./vocab.json",
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
)

print("✓ Tokenizer Created")

# ------------------------------------------------------------
# FEATURE EXTRACTOR
# ------------------------------------------------------------

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True
)

print("✓ Feature Extractor Created")

# ------------------------------------------------------------
# PROCESSOR
# ------------------------------------------------------------

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
)

print("✓ Processor Created")

# ------------------------------------------------------------
# MODEL
# ------------------------------------------------------------

model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    vocab_size=len(tokenizer),
    pad_token_id=tokenizer.pad_token_id,
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,
    ignore_mismatched_sizes=True,
)

# Freeze CNN Feature Encoder
model.freeze_feature_encoder()

# ------------------------------------------------------------
# DEVICE
# ------------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

print("✓ Model Loaded")
print("Device :", device)

print("=" * 70)

# ============================================================
# VERIFY
# ============================================================

print("Model Vocabulary :", model.config.vocab_size)
print("Tokenizer Vocabulary :", len(tokenizer))

print("\nReturn Attention Mask :",
      processor.feature_extractor.return_attention_mask)

sample_text = dataset["train"][0]["transcription"].lower()

print("\nSample Text:")
print(sample_text)

token_ids = processor(
    text=sample_text,
    add_special_tokens=False
).input_ids

print("\nFirst 30 Token IDs:")
print(token_ids[:30])

print("\nDecoded Text:")
print(
    processor.decode(
        token_ids,
        group_tokens=False
    )
)

print("=" * 70)

CREATING TOKENIZER, PROCESSOR & MODEL
✓ Tokenizer Created
✓ Feature Extractor Created
✓ Processor Created


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/380M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/380M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     | 
-----------------------------+------------+-
quantizer.codevectors        | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
lm_head.bias                 | MISSING    | 
lm_head.weight               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✓ Model Loaded
Device : cuda
Model Vocabulary : 32
Tokenizer Vocabulary : 32

Return Attention Mask : True

Sample Text:
a tornado is a spinning column of very low-pressure air which sucks the surrounding air inward and upward

First 30 Token IDs:
[2, 0, 21, 16, 19, 15, 2, 5, 16, 0, 10, 20, 0, 2, 0, 20, 17, 10, 15, 15, 10, 15, 8, 0, 4, 16, 13, 22, 14, 15]

Decoded Text:
a tornado is a spinning column of very low[UNK]pressure air which sucks the surrounding air inward and upward


In [9]:
print(processor.feature_extractor.return_attention_mask)

True


In [10]:

# ============================================================
# PART 5 - PREPARE DATASET
# ============================================================

import numpy as np

print("=" * 70)
print("PREPARING DATASET")
print("=" * 70)


def prepare_dataset(batch):

    # -------------------------
    # Audio
    # -------------------------
    audio = batch["audio"]

    speech = np.asarray(
        audio["array"],
        dtype=np.float32
    )

    speech = np.nan_to_num(speech)

    batch["input_values"] = processor(
        speech,
        sampling_rate=16000
    ).input_values[0]

    batch["input_length"] = len(batch["input_values"])

    # -------------------------
    # Text
    # -------------------------
    text = batch["transcription"].lower()

    # Keep only characters موجود in vocabulary
    text = "".join(
        ch if ch in processor.tokenizer.get_vocab() else " "
        for ch in text
    )

    # Space -> |
    text = text.replace(" ", "|")

    batch["labels"] = processor(
        text=text,
        add_special_tokens=False
    ).input_ids

    return batch


# ------------------------------------------------------------
# Apply preprocessing
# ------------------------------------------------------------

train_dataset = dataset["train"].map(
    prepare_dataset,
    remove_columns=dataset["train"].column_names,
    desc="Preparing Train Dataset"
)

validation_dataset = dataset["validation"].map(
    prepare_dataset,
    remove_columns=dataset["validation"].column_names,
    desc="Preparing Validation Dataset"
)

test_dataset = dataset["test"].map(
    prepare_dataset,
    remove_columns=dataset["test"].column_names,
    desc="Preparing Test Dataset"
)

print("=" * 70)
print("DATASET READY")
print("=" * 70)

print("Train      :", len(train_dataset))
print("Validation :", len(validation_dataset))
print("Test       :", len(test_dataset))

print("\nColumns:")
print(train_dataset.column_names)

print("=" * 70)

PREPARING DATASET


Preparing Train Dataset:   0%|          | 0/2602 [00:00<?, ? examples/s]

Preparing Validation Dataset:   0%|          | 0/394 [00:00<?, ? examples/s]

Preparing Test Dataset:   0%|          | 0/647 [00:00<?, ? examples/s]

DATASET READY
Train      : 2602
Validation : 394
Test       : 647

Columns:
['input_values', 'input_length', 'labels']


In [11]:
sample = train_dataset[0]

print(sample.keys())

print("\nInput Length :", len(sample["input_values"]))
print("Label Length :", len(sample["labels"]))

print("\nFirst 30 Labels:")
print(sample["labels"][:30])

print("\nDecoded Labels:")
print(
    processor.decode(
        sample["labels"],
        group_tokens=False
    )
)

dict_keys(['input_values', 'input_length', 'labels'])

Input Length : 108800
Label Length : 105

First 30 Labels:
[2, 0, 21, 16, 19, 15, 2, 5, 16, 0, 10, 20, 0, 2, 0, 20, 17, 10, 15, 15, 10, 15, 8, 0, 4, 16, 13, 22, 14, 15]

Decoded Labels:
a tornado is a spinning column of very low pressure air which sucks the surrounding air inward and upward


In [12]:
# ============================================================
# PART 6 - DATA COLLATOR
# ============================================================

from dataclasses import dataclass
from typing import Dict, List, Union

import torch

print("=" * 70)
print("CREATING DATA COLLATOR")
print("=" * 70)


@dataclass
class DataCollatorCTCWithPadding:

    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features):

        # -------------------------
        # Input Features
        # -------------------------
        input_features = [
            {"input_values": feature["input_values"]}
            for feature in features
        ]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )

        # -------------------------
        # Labels
        # -------------------------
        label_features = [
            {"input_ids": feature["labels"]}
            for feature in features
        ]

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt",
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1),
            -100,
        )

        batch["labels"] = labels

        return batch


# ------------------------------------------------------------
# Create Data Collator
# ------------------------------------------------------------

data_collator = DataCollatorCTCWithPadding(
    processor=processor,
    padding=True
)

print("✓ Data Collator Created Successfully")

print("=" * 70)

CREATING DATA COLLATOR
✓ Data Collator Created Successfully


In [13]:
sample_batch = data_collator(
    [
        train_dataset[0],
        train_dataset[1]
    ]
)

print(sample_batch.keys())

print("\nInput Shape :")
print(sample_batch["input_values"].shape)

print("\nLabels Shape :")
print(sample_batch["labels"].shape)

print("\nAttention Mask Shape :")
print(sample_batch["attention_mask"].shape)

KeysView({'input_values': tensor([[ 2.7803e-05,  2.7803e-05,  2.7803e-05,  ...,  0.0000e+00,
          0.0000e+00,  0.0000e+00],
        [ 1.9504e-04,  1.9504e-04,  1.9504e-04,  ..., -2.6172e-03,
          2.2810e-04,  3.1527e-03]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1]], dtype=torch.int32), 'labels': tensor([[   2,    0,   21,   16,   19,   15,    2,    5,   16,    0,   10,   20,
            0,    2,    0,   20,   17,   10,   15,   15,   10,   15,    8,    0,
            4,   16,   13,   22,   14,   15,    0,   16,    7,    0,   23,    6,
           19,   26,    0,   13,   16,   24,    0,   17,   19,    6,   20,   20,
           22,   19,    6,    0,    2,   10,   19,    0,   24,    9,   10,    4,
            9,    0,   20,   22,    4,   12,   20,    0,   21,    9,    6,    0,
           20,   22,   19,   19,   16,   22,   15,    5,   10,   15,    8,    0,
            2,   10,   19,    0,   10,   15,   24,    2,   19,    5,    0,    2,
 

In [14]:
# ============================================================
# PART 7 - WORD ERROR RATE (WER)
# ============================================================

import evaluate
import numpy as np

print("=" * 70)
print("LOADING WER METRIC")
print("=" * 70)

wer_metric = evaluate.load("wer")

print("WER Metric Loaded Successfully")

print("=" * 70)


def compute_metrics(pred):

    # Model Predictions
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)

    # Labels
    label_ids = pred.label_ids.copy()

    # Replace ignored index (-100) with PAD token
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    # Decode predictions
    pred_str = processor.batch_decode(pred_ids)

    # Decode references
    label_str = processor.batch_decode(
        label_ids,
        group_tokens=False
    )

    # Compute WER
    wer = wer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    return {
        "wer": wer
    }


print("compute_metrics() Ready")

print("=" * 70)

LOADING WER METRIC


WER Metric Loaded Successfully
compute_metrics() Ready


In [15]:
!pip install -q -U transformers==4.57.1 datasets evaluate jiwer accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 108.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 40.3 MB/s eta 0:00:00


In [16]:
import torch

In [17]:
# ============================================================
# PART 8 - TRAINING ARGUMENTS & TRAINER
# ============================================================

from transformers import TrainingArguments, Trainer

print("="*70)
print("CREATING TRAINER")
print("="*70)

training_args = TrainingArguments(

    output_dir="./wav2vec2-fleurs-en",

    # Training
    do_train=True,
    do_eval=True,

    num_train_epochs=30,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    gradient_accumulation_steps=2,

    learning_rate=1e-4,
    warmup_ratio=0.10,
    weight_decay=0.005,
    max_grad_norm=1.0,

    fp16=torch.cuda.is_available(),

    # Logging
    logging_strategy="steps",
    logging_steps=25,

    # Evaluation
    eval_strategy="epoch",

    save_strategy="epoch",

    save_total_limit=2,

    load_best_model_at_end=True,

    metric_for_best_model="wer",
    greater_is_better=False,

    report_to="none",

    remove_unused_columns=False,

    dataloader_num_workers=2,

    group_by_length=True,
)

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=validation_dataset,

    processing_class=processor,

    data_collator=data_collator,

    compute_metrics=compute_metrics,
)

print("Trainer Created Successfully")
print("="*70)

CREATING TRAINER


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Trainer Created Successfully


In [ ]:
# ============================================================
# PART 9 - TRAIN MODEL
# ============================================================

print("=" * 70)
print("START TRAINING")
print("=" * 70)

trainer.train()

print("=" * 70)
print("TRAINING COMPLETED")
print("=" * 70)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 31, 'bos_token_id': 30}.


START TRAINING


Epoch,Training Loss,Validation Loss,Wer
1,11.954150,5.926311,1.000000
2,11.575872,5.872033,1.000000
3,5.634725,2.229967,0.715863
4,2.280766,1.325920,0.490615
5,1.689104,1.167914,0.413324
6,1.117494,1.194625,0.390382
7,1.045106,1.233952,0.374678
8,0.709322,1.202687,0.353699
9,0.701668,1.206470,0.336891
10,0.553993,1.165518,0.325972


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# ============================================================
# PART 10 - TEST EVALUATION
# ============================================================

print("=" * 70)
print("EVALUATING ON TEST SET")
print("=" * 70)

test_results = trainer.evaluate(test_dataset)

print("\nTest Results")
print(test_results)

print("=" * 70)

In [ ]:
# ============================================================
# PART 11 - SAVE MODEL
# ============================================================

SAVE_PATH = "./wav2vec2_fleurs_english"

trainer.save_model(SAVE_PATH)
processor.save_pretrained(SAVE_PATH)

print("Model Saved Successfully")
print("Location :", SAVE_PATH)

In [ ]:
# ============================================================
# PART 12 - LOAD SAVED MODEL
# ============================================================

from transformers import (
    Wav2Vec2Processor,
    Wav2Vec2ForCTC
)

processor = Wav2Vec2Processor.from_pretrained(
    "./wav2vec2_fleurs_english"
)

model = Wav2Vec2ForCTC.from_pretrained(
    "./wav2vec2_fleurs_english"
)

model.to(device)

print("Model Loaded Successfully")

In [ ]:
# ============================================================
# PART 13 - SINGLE AUDIO PREDICTION
# ============================================================

import torch

sample = test_dataset[0]

input_values = torch.tensor(
    sample["input_values"]
).unsqueeze(0).to(device)

attention_mask = torch.ones_like(input_values)

with torch.no_grad():

    logits = model(
        input_values,
        attention_mask=attention_mask
    ).logits

pred_ids = torch.argmax(logits, dim=-1)

prediction = processor.batch_decode(pred_ids)[0]

print("=" * 70)
print("GROUND TRUTH")
print(processor.decode(sample["labels"], group_tokens=False))

print("\nPREDICTION")
print(prediction)

print("=" * 70)

In [ ]:
# ============================================================
# PART 14 - MULTIPLE TEST PREDICTIONS
# ============================================================

import random
import torch

print("=" * 90)

indices = random.sample(range(len(test_dataset)), 10)

for i in indices:

    sample = test_dataset[i]

    input_values = torch.tensor(
        sample["input_values"]
    ).unsqueeze(0).to(device)

    attention_mask = torch.ones_like(input_values)

    with torch.no_grad():

        logits = model(
            input_values,
            attention_mask=attention_mask
        ).logits

    pred_ids = torch.argmax(logits, dim=-1)

    prediction = processor.batch_decode(pred_ids)[0]

    target = processor.decode(
        sample["labels"],
        group_tokens=False
    )

    print("-" * 90)
    print("Target     :", target)
    print("Prediction :", prediction)

print("=" * 90)

In [ ]:
# ============================================================
# PART 15 - TEST EVALUATION
# ============================================================

import torch
import pandas as pd
from jiwer import wer

model.eval()

predictions = []
references = []

for sample in test_dataset:

    input_values = torch.tensor(
        sample["input_values"],
        dtype=torch.float32
    ).unsqueeze(0).to(device)

    attention_mask = torch.ones_like(input_values)

    with torch.no_grad():
        logits = model(
            input_values=input_values,
            attention_mask=attention_mask
        ).logits

    pred_ids = torch.argmax(logits, dim=-1)

    prediction = processor.batch_decode(pred_ids)[0].lower().strip()

    labels = sample["labels"]
    labels = [l for l in labels if l != -100]

    reference = processor.decode(
        labels,
        group_tokens=False
    ).lower().strip()

    predictions.append(prediction)
    references.append(reference)

test_wer = wer(references, predictions)

print("="*70)
print("FINAL TEST WER :", round(test_wer,4))
print("="*70)

In [ ]:
# ============================================================
# PART 16 - RANDOM TEST PREDICTIONS
# ============================================================

import random
import pandas as pd

idx = random.sample(range(len(predictions)),10)

rows=[]

for i in idx:

    rows.append({

        "Sample":i,

        "Ground Truth":references[i],

        "Prediction":predictions[i]

    })

results_df = pd.DataFrame(rows)

results_df

In [ ]:
# ============================================================
# PART 17 - LOSS GRAPH
# ============================================================

import matplotlib.pyplot as plt

logs = trainer.state.log_history

train_loss = []
eval_loss = []
epochs = []

for log in logs:

    if "eval_loss" in log:

        epochs.append(log["epoch"])

        eval_loss.append(log["eval_loss"])

    if "loss" in log:

        train_loss.append(log["loss"])

plt.figure(figsize=(8,5))

plt.plot(range(1,len(train_loss)+1),train_loss,label="Training Loss")

plt.plot(epochs,eval_loss,label="Validation Loss")

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.title("Training vs Validation Loss")

plt.legend()

plt.grid(True)

plt.show()

In [ ]:
# ============================================================
# PART 18 - WER GRAPH
# ============================================================

wer_values=[]
epochs=[]

for log in trainer.state.log_history:

    if "eval_wer" in log:

        wer_values.append(log["eval_wer"])

        epochs.append(log["epoch"])

plt.figure(figsize=(8,5))

plt.plot(epochs,wer_values,marker="o")

plt.xlabel("Epoch")

plt.ylabel("WER")

plt.title("Validation WER")

plt.grid(True)

plt.show()

In [ ]:
# ============================================================
# PART 19 - FINAL RESULTS TABLE
# ============================================================

import pandas as pd

best_log = None

for log in trainer.state.log_history:

    if "eval_wer" in log:

        best_log = log

results = pd.DataFrame({

    "Dataset":["FLEURS English"],

    "Model":["facebook/wav2vec2-base-960h"],

    "Epochs":[training_args.num_train_epochs],

    "Learning Rate":[training_args.learning_rate],

    "Batch Size":[training_args.per_device_train_batch_size],

    "Validation Loss":[round(best_log["eval_loss"],4)],

    "Validation WER":[round(best_log["eval_wer"],4)],

    "Test WER":[round(test_wer,4)]

})

results